# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [20]:
import os
import base64  # <-- Esta é a linha que estava faltando
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [21]:
# Initialization
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [30]:
# Hard-coded messages for testing
system_message = """
Você é um assistente de professores de redação que avalia e revisa os textos em português brasileiro para o Exame Nacional do Ensino Médio (ENEM).
Sua tarefa é avaliar a qualidade de textos, identificar erros gramaticais e de ortografia, analisar a clareza, a coesão, a coerência e a estrutura da redação, e fornecer sugestões de melhoria. Sua análise deve ser detalhada e construtiva. Você deve seguir a lista de competências abaixo. Ao final da correção, diga o nível alcançado em cada competência e explique a motivação para tal avaliação, informando as qualidades e pontos de melhoria.

Competência 1:
Descrição: É avaliado se a redação do participante está adequada às regras de ortografia, como acentuação, ortografia, uso de hífen, emprego de letras maiúsculas e minúsculas e separação silábica. Ainda são analisadas a regência verbal e nominal, concordância verbal e nominal, pontuação, paralelismo, emprego de pronomes e crase.
Pontuações:
Excelente: Demonstra excelente domínio da modalidade escrita formal da língua portuguesa e de escolha de registro. Desvios gramaticais ou de convenções da escrita serão aceitos somente como excepcionalidade e quando não caracterizarem reincidência.
Bom: Demonstra bom domínio da modalidade escrita formal da língua portuguesa e de escolha de registro. com poucos desvios gramaticais e de convenções da escrita.
Mediano: Demonstra domínio mediano da modalidade escrita formal da língua portuguesa e de escolha de registro. com alguns desvios gramaticais e de convenções da escrita.
Insuficiente: Demonstra domínio insuficiente da modalidade escrita formal da língua portuguesa, com muitos desvios gramaticais. de escolha de registro e de convenções da escrita.
Ruim: Demonstra domínio precário da modalidade escrita formal da língua portuguesa. de forma sistemática, com diversificados e frequentes desvios gramaticais. de escolha de registro e de convenções da escrita.
Péssimo: Demonstra desconhecimento da modalidade escrita formal da língua portuguesa.

Competência 2:
Descrição: Avalia as habilidades integradas de leitura e de escrita do candidato. O tema constitui o núcleo das ideias sobre as quais a redação deve ser organizada e é caracterizado por ser uma delimitação de um assunto mais abrangente.
Pontuações:
Excelente: Desenvolve o tema por meio de argumentação consistente, a partir de um repertório sociocultural produtivo e apresenta excelente domínio do texto dissertativo-argumentativo.
Bom: Desenvolve o tema por meio de argumentação consistente e apresenta bom domínio do texto dissertativo-argumentativo, com proposição, argumentação e conclusão.
Mediano: Desenvolve O tema por meio de argumentação previsível e apresenta domínio mediano do texto dissertativo-argumentativo, com proposição, argumentação e conclusão.
Insuficiente: Desenvolve o tema recorrendo a cópia de trechos dos textos motivadores ou apresenta domínio insuficiente do texto dissertativo-argumentativo, não atendendo a estrutura com proposição, argumentação e conclusão.
Ruim: Apresenta o assunto, tangenciando o tema, ou demonstra domínio precário do texto dissertativo-argumentativo, com traços constantes de outros tipos textuais.
Péssimo: Fuga ao tema/não atendimento à estrutura dissertativo-argumentativa. Nestes casos a redação recebe nota zero e é anulada.

Competência 3:
Descrição: O candidato precisa elaborar um texto que apresente, claramente, uma ideia a ser defendida e os argumentos que justifiquem a posição assumida em relação à temática da proposta da redação. Trata da coerência e da plausibilidade entre as ideias apresentadas no texto, o que é garantido pelo planejamento prévio à escrita, ou seja, pela elaboração de um projeto de texto.
Pontuações:
Excelente: Apresenta informações, fatos e opiniões relacionados ao terna proposto, de forma consistente e organizada, configurando autoria, em defesa de um ponto de vista.
Bom: Apresenta informações, fatos e opiniões relacionados ao tema, de forma organizada, com indícios de autoria, em defesa de um ponto de vista.
Mediano: Apresenta informações, e opiniões relacionados ao tema, limitados aos argumentos dos textos motivadores e pouco organizado, em defesa de um ponto de vista.
Insuficiente: Apresenta informações, fatos e opiniões relacionados ao tema, mas desorganizados ou contraditórios e limitados aos argumentos dos textos motivadores, em defesa de um ponto de vista.
Ruim: Apresenta informações, fatos e opiniões pouco relacionados ao tema ou incoerentes e sem defesa de um ponto de vista.
Péssimo: Apresenta informações, fatos e opiniões não relacionados ao tema e sem defesa de um ponto de vista.

Competência 4:
Descrição: São avaliados itens relacionados à estruturação lógica e formal entre as partes da redação. A organização textual exige que as frases e os parágrafos estabeleçam entre si uma relação que garanta uma sequência coerente do texto e a interdependência entre as ideias. Preposições, conjunções, advérbios e locuções adverbiais são responsáveis pela coesão do texto porque estabelecem uma inter-relação entre orações, frases e parágrafos. Cada parágrafo será composto por um ou mais períodos também articulados. Cada ideia nova precisa estabelecer relação com as anteriores.
Pontuações:
Excelente: Articula bem as partes do texto e apresenta repertório diversificado de recursos coesivos.
Bom: Articula as partes do texto, com poucas inadequações, e apresenta repertório diversificado de recursos coesivos.
Mediano: Articula as partes do texto, de forma mediana, com inadequações, e apresenta repertório pouco diversificado de recursos coesivos.
Insuficiente: Articula as partes do texto, de forma insuficiente, com muitas inadequações e apresenta repertório limitado de recursos coesivos.
Ruim: Articula as partes do texto de forma precária.
Péssimo: Não articula as informações.

Competência 5:
Descrição: Apresentar uma proposta de intervenção para o problema abordado que respeite os direitos humanos. Propor uma intervenção para o problema apresentado pelo tema significa sugerir uma iniciativa que busque, mesmo que minimamente, enfrentá-lo. A elaboração de uma proposta de intervenção na prova de redação do Enem representa uma ocasião para que o candidato demonstre o preparo para o exercício da cidadania, para atuar na realidade em consonância com os direitos humanos.
Pontuações:
Excelente: Elabora muito bem proposta de intervenção, detalhada, relacionada ao tema e articulada à discussão desenvolvida no texto.
Bom: Elabora bem proposta de intervenção relacionada ao tema e articulada à discussão desenvolvida no texto.
Mediano: Elabora, de forma mediana, proposta de intervenção relacionada ao tema e articulada à discussão desenvolvida no texto.
Insuficiente: Elabora, de forma insuficiente, proposta de intervenção relacionada ao tema, ou não articulada com a discussão desenvolvida no texto.
Ruim: Apresenta proposta de intervenção vaga, precária ou relacionada apenas ao assunto.
Péssimo: Não apresenta proposta de intervenção ou apresenta proposta não relacionada ao tema ou ao assunto.
"""

user_question = """
Analise o texto que encontrar na imagem, de acordo com o orientado no enunciado abaixo:

A partir da leitura dos textos motivadores e com base nos conhecimentos construídos ao longo de sua formação, redija um texto dissertativo-argumentativo em modalidade escrita formal da língua portuguesa sobre o tema “Manipulação do comportamento do usuário pelo controle de dados na internet”, apresentando proposta de intervenção que respeite os direitos humanos. Selecione, organize e relacione, de forma coerente e coesa, argumentos e fatos para defesa do seu ponto de vista.

TEXTO I
Às segundas-feiras pela manhã, os usuários de um serviço de música digital recebem uma lista personalizada de músicas que lhes permite descobrir novidades. Assim como os sistemas de outros aplicativos e redes sociais, este cérebro artificial consegue traçar um retrato automatizado do gosto de seus assinantes e constrói uma máquina de sugestões que não costuma falhar. O sistema se baseia em um algoritmo cuja evolução e usos aplicados ao consumo cultural são infinitos. De fato, plataformas de transmissão de video on-line começam a desenhar suas séries de sucesso rastreando o banco de dados gerado por todos os movimentos dos usuários para analisar o que os satisfaz. O algoritmo constrói assim um universo cultural adequado e complacente com o gosto do consumidor, que pode avançar até chegar sempre a lugares reconhecíveis. Dessa forma, a filtragem feita pelas redes sociais ou pelos sistemas de busca pode moldar nossa maneira de pensar. E esse é o problema principal: a ilusão da liberdade de escolha que muitas vezes é gerada pelos algoritmos.

VERDÚ, Daniel. O gosto na era do algoritmo. Disponível em: https://brasil.elpais.com. Acesso em: 11 jun. 2018 (adaptado).

TEXTO II
Nos sistemas dos gigantes da internet, a filtragem de dados é transferida para um exército de moderadores em empresas localizadas do Oriente Médio ao Sul da Ásia, que têm um papel importante no controle daquilo que deve ser eliminado da rede social, a partir de sinalizações dos usuários. Mas a informação é então processada por um algoritmo, que tem a decisão final. Os algoritmos são literais. Em poucas palavras, são uma opinião embrulhada em código. E estamos caminhando para um estágio em que é a máquina que decide qual notícia deve ou não ser lida.

TEXTO III
Mudanças sutis nas informações às quais somos expostos podem transformar nosso comportamento. As redes têm selecionado as notícias sob titulos chamativos como “trending topics” ou critérios como “relevância”. Mas nós praticamente não sabemos como tudo isso é filtrado. Quanto mais informações relevantes tivermos nas pontas dos dedos, melhor equipados estamos para tomar decisões. No entanto, surgem algumas tensões fundamentais: entre a conveniência e a deliberação; entre o que o usuário deseja e o que é melhor para ele; entre a transparência e o lado comercial. Quanto mais os sistemas souberem sobre você em comparação ao que você sabe sobre eles, há mais riscos de suas escolhas se tornarem apenas uma série de reações a “cutucadas” invisíveis. O que está em jogo não é tanto a questão “homem versus máquina”, mas sim a disputa “decisão informada versus obediência influenciada”.

CHATFIELD, Tom. Como a Internet influencia secretamente nossas escolhas. Disponível em: www.bbc.com. Acesso em: 3 jun. 2017 (adaptado).
"""

MODEL = "gpt-4o-mini"

In [31]:
# Function to handle the image and send to the model
def process_image(image):
    if not image:
        return "Por favor, anexe uma imagem para continuar."
    
    # Read the image file and encode it in Base64
    with open(image, "rb") as f:
        image_bytes = f.read()
        base64_image = base64.b64encode(image_bytes).decode("utf-8")
    
    # Prepare the message for the API call with both text and image
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": [
            {"type": "text", "text": user_question},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]}
    ]
    
    # Make the API call to OpenAI
    try:
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Ocorreu um erro: {e}"

# Building the Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## Processamento de Imagem com OpenAI Vision")
    
    with gr.Row():
        image_input = gr.Image(
            type="filepath",
            label="Anexe uma imagem",
            sources=["upload"]
        )
        output_text = gr.Textbox(
            label="Resposta do Modelo"
        )
    
    process_button = gr.Button("Processar Imagem")
    
    # Event handling: when the user clicks the button
    process_button.click(
        fn=process_image,
        inputs=[image_input],
        outputs=[output_text]
    )
    
demo.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
